In [33]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
from fns import  _xlsx_1607_i3_GET_all_accounts

df_all = _xlsx_1607_i3_GET_all_accounts(
    data_folder="data",
    file_template="finances_{year}.xlsx",
    start_year=2021,
    end_year=2026,
    output_filename="finances_2021_2026.xlsx",
    output_sheet="ALL_TRANSACTIONS",
)

Processing yearly workbooks:   0%|          | 0/6 [00:00<?, ?workbook/s]


finances_2021.xlsx
Account sheets found: ['PayPal', 'OUCU Checking', 'OUCU Visa', 'PNC Joint']


Reading finances_2021.xlsx:   0%|          | 0/4 [00:00<?, ?sheet/s]

ADDED: PayPal | 40 rows
ADDED: OUCU Checking | 576 rows
ADDED: OUCU Visa | 36 rows
ADDED: PNC Joint | 6 rows

finances_2022.xlsx
Account sheets found: ['PayPal', 'OUCU Checking', 'OUCU Visa', 'PNC Joint', 'Robinhood']


Reading finances_2022.xlsx:   0%|          | 0/5 [00:00<?, ?sheet/s]

ADDED: PayPal | 83 rows
ADDED: OUCU Checking | 211 rows
ADDED: OUCU Visa | 25 rows
ADDED: PNC Joint | 565 rows
ADDED: Robinhood | 53 rows

finances_2023.xlsx
Account sheets found: ['PayPal', 'OUCU Checking', 'OUCU Visa', 'PNC Joint', 'Robinhood']


... (4 outputs left out)

Reading finances_2025.xlsx:   0%|          | 0/7 [00:00<?, ?sheet/s]

ADDED: PayPal | 653 rows
ADDED: OUCU Checking | 1,158 rows
ADDED: OUCU Visa | 94 rows
ADDED: Venmo | 87 rows
ADDED: PNC Joint | 23 rows
ADDED: Robinhood | 48 rows
ADDED: Cash App | 16 rows

finances_2026.xlsx
Account sheets found: ['PayPal', 'OUCU Checking', 'OUCU Visa', 'Venmo', 'PNC Joint', 'Robinhood', 'Chase Checking', 'Chase Savings', 'Chase Freedom']


Reading finances_2026.xlsx:   0%|          | 0/9 [00:00<?, ?sheet/s]

ADDED: PayPal | 116 rows
ADDED: OUCU Checking | 543 rows
ADDED: OUCU Visa | 24 rows
ADDED: Venmo | 15 rows
ADDED: PNC Joint | 10 rows
ADDED: Robinhood | 45 rows
ADDED: Chase Checking | 38 rows
ADDED: Chase Savings | 3 rows
ADDED: Chase Freedom | 28 rows

================ COMPLETE ================
Account sheets combined: 35
Total rows: 6,488
Total columns: 24
Output sheet count: 1
Saved to: /Users/yerik/_apple_lib/_a_progs/_a0ds_env/__finances/_2_accounts /checkings/data/finances_2021_2026.xlsx


In [9]:
# 0_FNS

import pandas as pd
from tqdm.auto import tqdm


# -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----######

def _balance_2007_i3_GET_period_table(
    df: pd.DataFrame,
    source_sheet: str | None = None,
    reference_date: str | None = "2026-07-01",
) -> pd.DataFrame:
    """
    Create a balance comparison table using:

        Reference date
        3 months before
        6 months before
        12 months before

    Every comparison date is set to the first day of its month.

    Balance rule:
        Use the final available transaction balance on or before
        each requested date.

    Parameters
    ----------
    df : pd.DataFrame
        Transaction dataframe.

    source_sheet : str | None
        Exact source_sheet name.

        When None, the function asks the user to select one.

    reference_date : str | None
        Reference date in YYYY-MM-DD format.

        Default:
            2026-07-01

        When None, the function asks the user and uses
        2026-07-01 when Enter is pressed.

    Returns
    -------
    pd.DataFrame
        Balance comparison table.
    """

    required_columns = {
        "date",
        "source_sheet",
        "balance",
    }

    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    df_work = df.copy()
    df_work["_original_order"] = range(len(df_work))

    # Clean dataframe.
    for step in tqdm(
        range(3),
        desc="Preparing balance data",
        unit="step",
    ):
        if step == 0:
            df_work["date"] = pd.to_datetime(
                df_work["date"],
                errors="coerce",
            )

        elif step == 1:
            df_work["balance"] = pd.to_numeric(
                df_work["balance"]
                .astype("string")
                .str.replace(",", "", regex=False)
                .str.replace("$", "", regex=False)
                .str.strip(),
                errors="coerce",
            )

        elif step == 2:
            df_work["source_sheet"] = (
                df_work["source_sheet"]
                .astype("string")
                .str.strip()
            )

            df_work = df_work.dropna(
                subset=[
                    "date",
                    "source_sheet",
                    "balance",
                ]
            )

    source_options = sorted(
        df_work["source_sheet"]
        .dropna()
        .unique()
        .tolist()
    )

    if not source_options:
        raise ValueError(
            "No source_sheet accounts were found."
        )

    # Ask for account.
    if source_sheet is None:
        print("\nSOURCE SHEETS")
        print("-" * 40)

        for number, option in enumerate(
            source_options,
            start=1,
        ):
            print(f"{number}. {option}")

        while True:
            selected_number = input(
                "\nSelect account number: "
            ).strip()

            try:
                selected_number = int(selected_number)

                if 1 <= selected_number <= len(source_options):
                    source_sheet = source_options[
                        selected_number - 1
                    ]
                    break

            except ValueError:
                pass

            print("Enter a valid account number.")

    if source_sheet not in source_options:
        raise ValueError(
            f"source_sheet '{source_sheet}' was not found."
        )

    # Ask for reference date when explicitly set to None.
    if reference_date is None:
        reference_date = input(
            "Reference date YYYY-MM-DD "
            "[default 2026-07-01]: "
        ).strip()

        if not reference_date:
            reference_date = "2026-07-01"

    reference_timestamp = pd.to_datetime(
        reference_date,
        errors="coerce",
    )

    if pd.isna(reference_timestamp):
        raise ValueError(
            "reference_date must use YYYY-MM-DD format."
        )

    # Force reference date to the first day of the month.
    reference_timestamp = (
        reference_timestamp
        .normalize()
        .replace(day=1)
    )

    df_account = (
        df_work.loc[
            df_work["source_sheet"] == source_sheet
        ]
        .sort_values(
            by=[
                "date",
                "_original_order",
            ]
        )
        .reset_index(drop=True)
    )

    target_periods = [
        {
            "period": "Reference",
            "months_before": 0,
        },
        {
            "period": "3 months before",
            "months_before": 3,
        },
        {
            "period": "6 months before",
            "months_before": 6,
        },
        {
            "period": "1 year before",
            "months_before": 12,
        },
    ]

    results = []

    for period_info in tqdm(
        target_periods,
        desc="Finding balances",
        unit="date",
    ):
        target_date = (
            reference_timestamp
            - pd.DateOffset(
                months=period_info["months_before"]
            )
        ).replace(day=1)

        eligible_rows = df_account.loc[
            df_account["date"] <= target_date
        ]

        if eligible_rows.empty:
            used_date = pd.NaT
            balance = pd.NA
        else:
            selected_row = eligible_rows.iloc[-1]

            used_date = selected_row["date"]
            balance = selected_row["balance"]

        results.append(
            {
                "source_sheet": source_sheet,
                "period": period_info["period"],
                "target_date": target_date,
                "balance_date_used": used_date,
                "balance": balance,
            }
        )

    df_results = pd.DataFrame(results)

    reference_balance = df_results.loc[
        df_results["period"] == "Reference",
        "balance",
    ].iloc[0]

    if pd.notna(reference_balance):
        df_results["difference_from_reference"] = (
            df_results["balance"]
            - reference_balance
        )
    else:
        df_results["difference_from_reference"] = pd.NA

    return df_results

In [10]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

df_balance_table = _balance_2007_i3_GET_period_table(
    df=df_all,
    source_sheet=None,
    reference_date="2026-07-01",
)

df_balance_table

Preparing balance data:   0%|          | 0/3 [00:00<?, ?step/s]


SOURCE SHEETS
----------------------------------------
1. Chase Checking
2. Chase Savings
3. OUCU Checking
4. PayPal



Select account number:  3


Finding balances:   0%|          | 0/4 [00:00<?, ?date/s]

,source_sheet,period,target_date,balance_date_used,balance,difference_from_reference
0,OUCU Checking,Reference,2026-07-01,2026-06-30,1005.39,0.00
1,OUCU Checking,3 months before,2026-04-01,2026-04-01,240.95,-764.44
2,OUCU Checking,6 months before,2026-01-01,2025-12-31,163.67,-841.72
3,OUCU Checking,1 year before,2025-07-01,2025-07-01,1467.89,462.50


In [15]:
df_account_type_freq = (
    df_all["account"]
    .value_counts(dropna=False)
    .rename_axis("account")
    .reset_index(name="frequency")
)

df_account_type_freq

,account,frequency
0,OUCU Checking,3435
1,PayPal — personal,1256
2,PNC Virtual Wallet Spend — JOINT (Soojin Kim &...,930
3,OUCU Platinum Visa,476
4,Robinhood brokerage (705587533),204
5,Venmo — personal (@lizardini),102
6,Chase Total Checking (x4197),38
7,Chase Freedom Visa (x3262),28
8,Cash App — personal (Yeriko),16
9,Chase Savings (x8635),3


In [14]:
df_all.columns

Index(['date', 'year', 'month', 'bank', 'account', 'account_type', 'direction',
       'amount', 'balance', 'currency', 'fees', 'category', 'subcategory',
       'counterparty', 'description', 'status', 'cross_case', 'txn_id',
       'source_file', 'source_year', 'source_sheet', 'sub_account', 'quantity',
       'unit_price'],
      dtype='object')

In [19]:
df_all.loc[df_all["account"].astype(str).str.strip().str.lower().eq("Chase Freedom Visa (x3262)	")].to_excel("chase_savings.xlsx", index=False)

In [23]:
df_1 = pd.read_excel("chase_freedom_visa.xlsx")

In [24]:
df_1

,date,year,month,bank,account,account_type,direction,amount,balance,currency,...,description,status,cross_case,txn_id,source_file,source_year,source_sheet,sub_account,quantity,unit_price
0,2026-03-21,2026,2026-03,Chase,Chase Freedom Visa (x3262),credit,out,-27.00,NaN,USD,...,& 12850 EXPRESS INC DEARBORN MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN
1,2026-03-21,2026,2026-03,Chase,Chase Freedom Visa (x3262),credit,out,-6.36,NaN,USD,...,& QAHWAH HOUSE DEARBORN MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN
2,2026-03-21,2026,2026-03,Chase,Chase Freedom Visa (x3262),credit,out,-20.13,NaN,USD,...,& MARCUS MARKET DETROIT MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN
3,2026-03-22,2026,2026-03,Chase,Chase Freedom Visa (x3262),credit,out,-8.32,NaN,USD,...,& SQ *ALBA Detroit MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN
4,2026-03-22,2026,2026-03,Chase,Chase Freedom Visa (x3262),credit,out,-12.02,NaN,USD,...,& FOXTOWN MARKET DETROIT MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN
5,2026-03-23,2026,2026-03,Chase,Chase Freedom Visa (x3262),credit,in,150.00,NaN,USD,...,SQ *L AND L GRACE Southfield MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN
6,2026-03-23,2026,2026-03,Chase,Chase Freedom Visa (x3262),credit,out,-150.00,NaN,USD,...,& SQ *L AND L GRACE Southfield MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN
7,2026-03-26,2026,2026-04,Chase,Chase Freedom Visa (x3262),credit,out,-10.00,NaN,USD,...,SPOTLITE DETROIT DETROIT MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN
8,2026-03-29,2026,2026-04,Chase,Chase Freedom Visa (x3262),credit,out,-6.68,NaN,USD,...,SQ *CAFE PRINCE Detroit MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN
9,2026-03-30,2026,2026-04,Chase,Chase Freedom Visa (x3262),credit,out,-7.79,NaN,USD,...,TST*CANNELLE PATISSERIE Detroit MI,Posted,NaN,NaN,finances_2026.xlsx,2026,Chase Freedom,NaN,NaN,NaN


In [22]:
df_all.loc[
    df_all["account"].astype(str).str.strip().str.casefold().eq(
        "Chase Freedom Visa (x3262)".casefold()
    )
].to_excel("chase_freedom_visa.xlsx", index=False)

In [26]:
#!/usr/bin/env python3
"""
Concatenate per-account .xlsx files into one master transaction table.

Run it, press Enter to accept the default folder (or paste a different one),
and it writes finances_master_rebuilt.xlsx into that same folder.

Requires: pandas, openpyxl   ->   pip install pandas openpyxl
"""
import sys
from pathlib import Path
import pandas as pd

DEFAULT_FOLDER = "/Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNTS_by_account"
OUTPUT_NAME = "finances_master_rebuilt.xlsx"


def get_folder() -> Path:
    if len(sys.argv) > 1:                       # allow: python concat_accounts.py "/some/folder"
        return Path(sys.argv[1]).expanduser()
    ans = input(f"Folder with the account files [{DEFAULT_FOLDER}]: ").strip()
    return Path(ans or DEFAULT_FOLDER).expanduser()


def find_date_col(df: pd.DataFrame):
    for c in df.columns:
        if str(c).strip().lower() == "date":
            return c
    return None


def main():
    folder = get_folder()
    if not folder.is_dir():
        print(f"Not a folder: {folder}")
        sys.exit(1)

    # every .xlsx under the folder, skipping the _to_delete stash, temp locks, and prior output
    files = [
        p for p in folder.rglob("*.xlsx")
        if "_to_delete" not in p.parts
        and not p.name.startswith("~$")
        and p.name != OUTPUT_NAME
    ]

    frames, skipped = [], []
    print(f"\nScanning: {folder}\n")
    for p in sorted(files):
        try:
            df = pd.read_excel(p)
        except Exception as e:
            skipped.append((p.name, f"read error: {e}"))
            continue
        dcol = find_date_col(df)
        if dcol is None:                        # e.g. Coinbase / PayPal-CAI raw exports have no 'date' header
            skipped.append((p.name, "no 'date' column"))
            continue
        df = df.rename(columns={dcol: "date"})
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["source_file"] = str(p.relative_to(folder))
        frames.append(df)
        print(f"  + {p.relative_to(folder)}  ({len(df)} rows)")

    if not frames:
        print("\nNo usable account files found (nothing had a 'date' column).")
        sys.exit(1)

    master = pd.concat(frames, ignore_index=True, sort=False)
    master = master.sort_values("date").reset_index(drop=True)

    before = len(master)
    master = master.drop_duplicates().reset_index(drop=True)
    dupes = before - len(master)

    master["date"] = pd.to_datetime(master["date"], errors="coerce").dt.strftime("%Y-%m-%d")

    out = folder / OUTPUT_NAME
    master.to_excel(out, index=False, sheet_name="ALL_TRANSACTIONS")

    print(f"\nMerged {len(frames)} files  ->  {len(master)} rows  ({dupes} exact duplicates removed)")
    if skipped:
        print("\nSkipped (no date column / unreadable) — check these by hand if needed:")
        for name, why in skipped:
            print(f"  - {name}: {why}")
    print(f"\nSaved: {out}")


if __name__ == "__main__":
    main()

Not a folder: -f


SystemExit: 1

In [27]:
# 0_FNS

#!/usr/bin/env python3

import sys
from pathlib import Path

import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    # Script still runs when tqdm is not installed.
    def tqdm(iterable, **kwargs):
        return iterable


DEFAULT_FOLDER = (
    "/Users/yerik/Documents/___Divorce_2nd_Paper/"
    "__inbox/ACCOUNTS_by_account"
)

OUTPUT_NAME = "finances_master_rebuilt.xlsx"

SKIP_SHEET_KEYWORDS = (
    "summary",
    "summaries",
    "dashboard",
    "pivot",
    "totals",
)


# -----######-----###### GET INPUT FOLDER -----######-----######

def _concat_2007_folder_GET_path() -> Path:
    """
    Get the account-files folder from a command-line argument
    or from user input.
    """

    if len(sys.argv) > 1:
        folder = sys.argv[1]
    else:
        folder = input(
            f"Folder with account files [{DEFAULT_FOLDER}]: "
        ).strip()

        if not folder:
            folder = DEFAULT_FOLDER

    return Path(folder).expanduser().resolve()


# -----######-----###### FIND DATE COLUMN -----######-----######

def _concat_2007_date_GET_column(
    df: pd.DataFrame,
) -> str | None:
    """
    Find the most likely transaction date column.
    """

    date_candidates = {
        "date",
        "transaction date",
        "transaction_date",
        "posted date",
        "posted_date",
        "activity date",
        "activity_date",
    }

    for column in df.columns:
        normalized = (
            str(column)
            .strip()
            .lower()
            .replace("-", " ")
        )

        if normalized in date_candidates:
            return column

    return None


# -----######-----###### CHECK SKIPPED SHEETS -----######-----######

def _concat_2007_sheet_GET_skip(
    sheet_name: str,
) -> bool:
    """
    Return True when the worksheet looks like a summary sheet.
    """

    normalized = str(sheet_name).strip().lower()

    return any(
        keyword in normalized
        for keyword in SKIP_SHEET_KEYWORDS
    )


# -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----######

def _concat_2007_i1_GET_master(
    folder_path: str | Path = DEFAULT_FOLDER,
    output_name: str = OUTPUT_NAME,
) -> pd.DataFrame:
    """
    Read all usable worksheets from every .xlsx file and create
    one master transaction workbook.

    Parameters
    ----------
    folder_path : str | Path
        Folder containing the per-account Excel files.

    output_name : str
        Name of the final merged workbook.

    Returns
    -------
    pd.DataFrame
        Combined master transaction table.
    """

    folder = Path(folder_path).expanduser().resolve()

    if not folder.is_dir():
        raise NotADirectoryError(
            f"Folder does not exist:\n{folder}"
        )

    output_path = folder / output_name

    # Find all Excel files while excluding output/temp/stash files.
    files = sorted(
        file_path
        for file_path in folder.rglob("*.xlsx")
        if file_path.resolve() != output_path.resolve()
        and not file_path.name.startswith("~$")
        and "_to_delete" not in {
            part.lower()
            for part in file_path.parts
        }
    )

    if not files:
        raise FileNotFoundError(
            f"No .xlsx files found inside:\n{folder}"
        )

    frames = []
    skipped = []
    loaded_sheets = 0

    print(f"\nScanning folder:\n{folder}\n")

    for file_path in tqdm(
        files,
        desc="Reading Excel files",
        unit="file",
    ):
        relative_file = file_path.relative_to(folder)

        try:
            excel_file = pd.ExcelFile(
                file_path,
                engine="openpyxl",
            )
        except Exception as error:
            skipped.append(
                {
                    "source_file": str(relative_file),
                    "source_sheet": None,
                    "reason": f"Workbook read error: {error}",
                }
            )
            continue

        for sheet_name in excel_file.sheet_names:

            if _concat_2007_sheet_GET_skip(sheet_name):
                skipped.append(
                    {
                        "source_file": str(relative_file),
                        "source_sheet": sheet_name,
                        "reason": "Summary sheet skipped",
                    }
                )
                continue

            try:
                df_sheet = pd.read_excel(
                    excel_file,
                    sheet_name=sheet_name,
                )
            except Exception as error:
                skipped.append(
                    {
                        "source_file": str(relative_file),
                        "source_sheet": sheet_name,
                        "reason": f"Sheet read error: {error}",
                    }
                )
                continue

            if df_sheet.empty:
                skipped.append(
                    {
                        "source_file": str(relative_file),
                        "source_sheet": sheet_name,
                        "reason": "Empty sheet",
                    }
                )
                continue

            # Clean column-name whitespace.
            df_sheet.columns = [
                str(column).strip()
                for column in df_sheet.columns
            ]

            date_column = _concat_2007_date_GET_column(
                df_sheet
            )

            if date_column is None:
                skipped.append(
                    {
                        "source_file": str(relative_file),
                        "source_sheet": sheet_name,
                        "reason": "No recognized date column",
                    }
                )
                continue

            if date_column != "date":
                df_sheet = df_sheet.rename(
                    columns={
                        date_column: "date",
                    }
                )

            df_sheet["date"] = pd.to_datetime(
                df_sheet["date"],
                errors="coerce",
            )

            # Remove fully blank rows.
            df_sheet = df_sheet.dropna(
                how="all"
            ).copy()

            # Keep provenance for every transaction.
            df_sheet["source_file"] = str(relative_file)
            df_sheet["source_sheet"] = str(sheet_name)

            frames.append(df_sheet)
            loaded_sheets += 1

            tqdm.write(
                f"+ {relative_file} | "
                f"{sheet_name} | "
                f"{len(df_sheet):,} rows"
            )

    if not frames:
        raise ValueError(
            "No usable worksheets were found. "
            "Every sheet was empty, unreadable, a summary, "
            "or missing a date column."
        )

    master = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )

    # Sort dated records first and undated records last.
    master = (
        master
        .sort_values(
            by="date",
            ascending=True,
            na_position="last",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    before_duplicates = len(master)

    # Removes only completely identical rows.
    master = (
        master
        .drop_duplicates()
        .reset_index(drop=True)
    )

    duplicates_removed = (
        before_duplicates - len(master)
    )

    # Put useful columns first when they exist.
    preferred_columns = [
        "date",
        "year",
        "month",
        "bank",
        "account",
        "account_type",
        "direction",
        "amount",
        "balance",
        "currency",
        "fees",
        "category",
        "subcategory",
        "counterparty",
        "description",
        "status",
        "cross_case",
        "txn_id",
        "source_file",
        "source_year",
        "source_sheet",
    ]

    first_columns = [
        column
        for column in preferred_columns
        if column in master.columns
    ]

    remaining_columns = [
        column
        for column in master.columns
        if column not in first_columns
    ]

    master = master[
        first_columns + remaining_columns
    ]

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
        date_format="YYYY-MM-DD",
        datetime_format="YYYY-MM-DD",
    ) as writer:

        master.to_excel(
            writer,
            index=False,
            sheet_name="ALL_TRANSACTIONS",
        )

        worksheet = writer.sheets["ALL_TRANSACTIONS"]

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        # Reasonable Excel column widths.
        for column_cells in worksheet.columns:
            column_letter = column_cells[0].column_letter

            max_length = max(
                len(str(cell.value))
                if cell.value is not None
                else 0
                for cell in column_cells[:500]
            )

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                max(max_length + 2, 12),
                45,
            )

        if skipped:
            df_skipped = pd.DataFrame(skipped)

            df_skipped.to_excel(
                writer,
                index=False,
                sheet_name="SKIPPED_SHEETS",
            )

            skipped_worksheet = writer.sheets[
                "SKIPPED_SHEETS"
            ]

            skipped_worksheet.freeze_panes = "A2"
            skipped_worksheet.auto_filter.ref = (
                skipped_worksheet.dimensions
            )

    print("\n" + "=" * 70)
    print(f"Files scanned:        {len(files):,}")
    print(f"Sheets loaded:        {loaded_sheets:,}")
    print(f"Master rows:          {len(master):,}")
    print(f"Duplicates removed:   {duplicates_removed:,}")
    print(f"Skipped sheets:       {len(skipped):,}")
    print(f"Saved workbook:       {output_path}")
    print("=" * 70)

    return master

In [29]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

if __name__ == "__main__":

    selected_folder = _concat_2007_folder_GET_path("/Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNTS_by_account")

    df_all = _concat_2007_i1_GET_master(
        folder_path=selected_folder,
        output_name=OUTPUT_NAME,
    )

TypeError: _concat_2007_folder_GET_path() takes 0 positional arguments but 1 was given

In [31]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

from pathlib import Path

selected_folder = Path(
    "/Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNTS_by_account"
).expanduser()

df_all2 = _concat_2007_i1_GET_master(
    folder_path=selected_folder,
    output_name=OUTPUT_NAME,
)


Scanning folder:
/Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNTS_by_account



Reading Excel files:   0%|          | 0/14 [00:00<?, ?file/s]

+ Cash_App_personal/Cash_App_personal.xlsx | Sheet1 | 18 rows
+ Chase_Freedom_Visa_x3262/Chase_Freedom_Visa_x3262.xlsx | Sheet1 | 41 rows
+ Chase_Savings_x8635/Chase_Savings_x8635.xlsx | Sheet1 | 4 rows
+ Chase_Total_Checking_x4197/Chase_Total_Checking_x4197.xlsx | Sheet1 | 58 rows
+ OUCU_Grizzly_Saver_03/OUCU_Grizzly_Saver_03.xlsx | Sheet1 | 108 rows
+ OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51.xlsx | Sheet1 | 476 rows
+ OUCU_Share_Secured_70/OUCU_Share_Secured_70.xlsx | Sheet1 | 94 rows
+ OUCU_Totally_Gold_Checking_83/OUCU_Totally_Gold_Checking_83.xlsx | Sheet1 | 3,249 rows
+ PNC_Joint_Spend_x8775/PNC_Joint_Spend_x8775.xlsx | Sheet1 | 931 rows
+ PayPal_personal/PayPal_personal.xlsx | Sheet1 | 1,256 rows
+ Robinhood_x705587533/Robinhood_x705587533.xlsx | Sheet1 | 204 rows
+ Venmo_lizardini/Venmo_lizardini.xlsx | Sheet1 | 102 rows

Files scanned:        14
Sheets loaded:        12
Master rows:          6,477
Duplicates removed:   64
Skipped sheets:       2
Saved workbook:       /Use

In [35]:
df_all2

,date,year,month,bank,account,account_type,direction,amount,balance,currency,...,description,status,cross_case,txn_id,source_file,source_year,source_sheet,sub_account,quantity,unit_price
0,2020-12-06,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-21.72,NaN,USD,...,24692160341100380975277 UNITEDHEALTHONE 800-65...,Posted,NaN,1206,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
1,2020-12-07,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-30.49,NaN,USD,...,24427330342730240632179 MARKET SQUARE BIRMINGH...,Posted,NaN,1208,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
2,2020-12-08,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-77.00,NaN,USD,...,Cash Advance ComputerLine Transfer To Share 83,Posted,NaN,1208,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
3,2020-12-08,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-11.72,NaN,USD,...,Cash Advance ComputerLine Transfer To Share 83,Posted,NaN,1208,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
4,2020-12-26,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-14.95,NaN,USD,...,24692160361100926597689 Audible*KO9HA2AP3 Amzn...,Posted,NaN,1226,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6472,2026-07-15,2026,7,Chase,Chase Total Checking (x4197),checking,out,-150.00,1576.76,USD,...,Zelle payment to Reynier,posted,NaN,NaN,Chase_Total_Checking_x4197/Chase_Total_Checkin...,2026,Sheet1,NaN,NaN,NaN
6473,2026-07-15,2026,7,OUCU,OUCU Totally Gold Checking (542096-83),checking,out,-32.92,708.57,USD,...,POS UBR* Uber,posted,NaN,NaN,OUCU_Totally_Gold_Checking_83/OUCU_Totally_Gol...,2026,Sheet1,NaN,NaN,NaN
6474,2026-07-15,2026,7,OUCU,OUCU Totally Gold Checking (542096-83),checking,out,-60.00,648.57,USD,...,ACH Withdrawal VENMO payment,posted,NaN,NaN,OUCU_Totally_Gold_Checking_83/OUCU_Totally_Gol...,2026,Sheet1,NaN,NaN,NaN
6475,2026-07-15,2026,7,OUCU,OUCU Totally Gold Checking (542096-83),checking,out,-291.97,356.6,USD,...,Card ••2906 online payment (biller not shown; ...,posted,NaN,NaN,OUCU_Totally_Gold_Checking_83/OUCU_Totally_Gol...,2026,Sheet1,NaN,NaN,NaN


In [34]:
len(df_all)

6488